# Run the below cell and the GUI will open

In [3]:
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
import torch
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

# Initialize the model
model = models.resnet18(pretrained=True)
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 2)
)

# Transforms for the input image
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Function to load the model weights dynamically
def load_model():
    global model
    model_path = filedialog.askopenfilename(filetypes=[("Model files", "*.pth")])
    if model_path:
        try:
            model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
            model.eval()
            messagebox.showinfo("Success", "Model loaded successfully!")
        except Exception as e:
            messagebox.showerror("Error", f"Failed to load model: {e}")

# Function to predict the image class
def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        outputs = model(image_tensor)
        _, predicted = torch.max(outputs, 1)
    return "Pneumonia detected :(" if predicted.item() == 1 else "Normal :)"

# Create the GUI
def create_gui():
    # Main window
    root = tk.Tk()
    root.title("Pneumonia Detection Model")

    # Function to load and display the image
    def show_image():
        global img_path, img_display
        img_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg;*.jpeg;*.png")])
        if img_path:
            img = Image.open(img_path)
            img = img.resize((300, 300))
            img_display = ImageTk.PhotoImage(img)
            panel.configure(image=img_display)
            panel.image = img_display

    # Function to analyze the image by the model
    def analyze_image():
        if not hasattr(model, 'state_dict'):
            messagebox.showerror("Error", "Please load a model first!")
            return
        if img_path:
            result = predict_image(img_path)
            result_label.config(text=f"Result: {result}")
        else:
            messagebox.showerror("Error", "Please load an image first!")

    # GUI frame
    panel = tk.Label(root)
    panel.pack(pady=20)

    # Load model button
    load_model_button = tk.Button(root, text="Load Model", command=load_model, height=2, width=15)
    load_model_button.pack(pady=5)

    # Upload image button
    load_button = tk.Button(root, text="Load X-Ray Image", command=show_image, height=2, width=15)
    load_button.pack(pady=5)

    # Analyze image button
    analyze_button = tk.Button(root, text="Analyze The Image", command=analyze_image, height=2, width=15)
    analyze_button.pack(pady=5)

    # Show result label
    result_label = tk.Label(root, text="Result: ", font=("Arial", 16))
    result_label.pack(pady=20)

    # Run the GUI
    root.mainloop()

# Run the GUI
create_gui()